# Fase 1 – Modelo Predictivo
## Student Performance & Study Habits Dataset

**Integrantes:**
- Brahian Ocampo Garcia
- Carlos Andres Pineda Ospina

**Curso:** Modelos y Simulación de Sistemas I — Andrés Parra — 2026-2

**Objetivo:** Construir un modelo de regresión para predecir el puntaje del examen final (`final_exam_score`) a partir de hábitos de estudio y factores demográficos.

**Dataset:** [Student Performance & Study Habits (Kaggle)](https://www.kaggle.com/datasets/harshadapatil31/student-performance-and-study-habits-dataset)

---

## 1. Importación de librerías

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib
import os

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


## 2. Carga y exploración inicial de los datos

> El archivo `student_performance_dataset.csv` está versionado en esta misma carpeta, de modo que el notebook se ejecuta sin descargas previas. Fuente original: [Kaggle](https://www.kaggle.com/datasets/harshadapatil31/student-performance-and-study-habits-dataset).

In [2]:
CSV_PATH = "student_performance_dataset.csv"

df = pd.read_csv(CSV_PATH)
print(f"Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head(10)

Dataset cargado: 1000 filas x 12 columnas


,student_id,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
0,1,Male,4.0,98.0,6.5,Bachelors,Yes,Yes,No,76.9,100.0,A
1,2,Female,6.3,100.0,5.7,High School,Yes,Yes,Yes,75.5,100.0,A
2,3,Male,4.9,85.3,7.9,Bachelors,Yes,No,Yes,88.5,97.3,A
3,4,Male,2.6,77.5,8.0,NaN,Yes,Yes,No,85.1,83.8,B
4,5,Male,2.2,89.6,4.6,Bachelors,Yes,No,Yes,61.8,68.3,D
5,6,Female,4.2,78.2,5.7,High School,Yes,No,No,79.8,81.6,B
6,7,Male,1.5,100.0,5.0,High School,Yes,Yes,Yes,58.6,69.6,D
7,8,Male,6.2,86.4,6.0,High School,Yes,Yes,No,78.4,88.0,B
8,9,Male,5.3,81.3,6.7,Bachelors,Yes,No,No,63.8,84.4,B
9,10,Female,2.8,86.8,5.1,Bachelors,Yes,Yes,No,57.7,74.3,C


In [3]:
# Tipos de datos y valores no nulos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   student_id                  1000 non-null   int64  
 1   gender                      1000 non-null   str    
 2   study_time_hours            1000 non-null   float64
 3   attendance_percent          1000 non-null   float64
 4   sleep_hours                 1000 non-null   float64
 5   parental_education          898 non-null    str    
 6   internet_access             1000 non-null   str    
 7   extracurricular_activities  1000 non-null   str    
 8   part_time_job               1000 non-null   str    
 9   previous_grade              1000 non-null   float64
 10  final_exam_score            1000 non-null   float64
 11  final_grade                 1000 non-null   str    
dtypes: float64(5), int64(1), str(6)
memory usage: 93.9 KB


In [4]:
# Estadísticas descriptivas — variables numéricas
df.describe().round(2)

,student_id,study_time_hours,attendance_percent,sleep_hours,previous_grade,final_exam_score
count,1000.00,1000.00,1000.00,1000.0,1000.00,1000.00
mean,500.50,3.57,85.09,6.8,69.74,83.54
std,288.82,1.48,9.27,1.2,12.61,10.34
min,1.00,0.50,54.80,3.2,31.30,46.80
25%,250.75,2.60,78.80,5.9,61.00,76.07
50%,500.50,3.60,85.20,6.8,69.60,83.80
75%,750.25,4.50,91.90,7.6,78.40,91.52
max,1000.00,8.10,100.00,10.0,100.00,100.00


In [5]:
# Estadísticas descriptivas — variables categóricas
df.describe(include='object')

,gender,parental_education,internet_access,extracurricular_activities,part_time_job,final_grade
count,1000,898,1000,1000,1000,1000
unique,2,4,2,2,2,5
top,Female,High School,Yes,Yes,No,B
freq,510,356,854,572,684,354


In [6]:
# Columnas del dataset
print("Columnas del dataset:")
for col in df.columns:
    print(f"  {col:30s} | tipo: {str(df[col].dtype):10s} | nulos: {df[col].isnull().sum()}")

Columnas del dataset:
  student_id                     | tipo: int64      | nulos: 0
  gender                         | tipo: str        | nulos: 0
  study_time_hours               | tipo: float64    | nulos: 0
  attendance_percent             | tipo: float64    | nulos: 0
  sleep_hours                    | tipo: float64    | nulos: 0
  parental_education             | tipo: str        | nulos: 102
  internet_access                | tipo: str        | nulos: 0
  extracurricular_activities     | tipo: str        | nulos: 0
  part_time_job                  | tipo: str        | nulos: 0
  previous_grade                 | tipo: float64    | nulos: 0
  final_exam_score               | tipo: float64    | nulos: 0
  final_grade                    | tipo: str        | nulos: 0


## 3. Análisis de valores faltantes

In [7]:
# Conteo y porcentaje de valores nulos por columna
nulos = df.isnull().sum()
porcentaje = (nulos / len(df)) * 100
tabla_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': porcentaje.round(2)})
tabla_nulos = tabla_nulos[tabla_nulos['Nulos'] > 0].sort_values('Porcentaje (%)', ascending=False)

if len(tabla_nulos) > 0:
    print("Variables con valores faltantes:")
    print(tabla_nulos)
    print(f"\nTotal de registros con al menos un nulo: {df.isnull().any(axis=1).sum()}")
else:
    print("No hay valores faltantes en el dataset.")

Variables con valores faltantes:
                    Nulos  Porcentaje (%)
parental_education    102            10.2

Total de registros con al menos un nulo: 102


**Observación:** aparecen 102 valores faltantes (10.2%) en `parental_education`. Antes de elegir una estrategia de tratamiento hay que saber de dónde salen, porque la estrategia correcta depende de qué significa ese vacío.

In [8]:
# ¿De dónde salen esos 102 nulos? Se lee el archivo sin la conversión automática de pandas
df_crudo = pd.read_csv(CSV_PATH, keep_default_na=False)

print("Valores reales de parental_education en el archivo:")
print(df_crudo['parental_education'].value_counts().to_string())
print(f"\nNulos en TODO el dataset leyéndolo sin conversión automática: {df_crudo.isnull().sum().sum()}")
print(f"'None' figura en la lista de nulos por defecto de pandas: {'None' in pd.io.parsers.readers.STR_NA_VALUES}")

Valores reales de parental_education en el archivo:
parental_education
High School    356
Bachelors      308
Masters        184
None           102
PhD             50

Nulos en TODO el dataset leyéndolo sin conversión automática: 0
'None' figura en la lista de nulos por defecto de pandas: True


**Conclusión:** el archivo no tiene ni un solo campo vacío. Los 102 «faltantes» son la cadena literal `None`, que pandas convierte a nulo por estar en su lista de valores nulos por defecto.

En esta variable `None` no significa dato ausente: es un nivel más de la escala educativa —`None`, `High School`, `Bachelors`, `Masters`, `PhD`—, es decir, padres sin educación formal. Es el peldaño de abajo.

**Estrategia de tratamiento:** conservar esos 102 registros como categoría propia y **no imputarlos**. Imputarlos con la moda asignaría `High School` a 102 estudiantes cuyos padres no tienen educación formal, destruyendo información válida y el 10.2% de una variable. Por la misma razón el pipeline no lleva imputador: verificado arriba, no hay ningún valor ausente que imputar.

In [9]:
# Se relee el dataset conservando 'None' como categoría
df = pd.read_csv(CSV_PATH, keep_default_na=False)

print(f"Nulos tras la relectura: {df.isnull().sum().sum()}")
print(f"Niveles de parental_education: {sorted(df['parental_education'].unique())}")

Nulos tras la relectura: 0
Niveles de parental_education: ['Bachelors', 'High School', 'Masters', 'None', 'PhD']


## 4. Análisis Exploratorio de Datos (EDA)

### 4.1 Distribución de la variable objetivo

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(df['final_exam_score'], bins=25, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Puntaje del examen final')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de final_exam_score')
axes[0].axvline(df['final_exam_score'].mean(), color='red', linestyle='--',
                label=f"Media: {df['final_exam_score'].mean():.1f}")
axes[0].axvline(df['final_exam_score'].median(), color='orange', linestyle='--',
                label=f"Mediana: {df['final_exam_score'].median():.1f}")
axes[0].legend()

# Boxplot
axes[1].boxplot(df['final_exam_score'], vert=True)
axes[1].set_ylabel('Puntaje del examen final')
axes[1].set_title('Boxplot de final_exam_score')

plt.tight_layout()
plt.show()

print(f"Media:      {df['final_exam_score'].mean():.2f}")
print(f"Mediana:    {df['final_exam_score'].median():.2f}")
print(f"Desv. est.: {df['final_exam_score'].std():.2f}")
print(f"Mínimo:     {df['final_exam_score'].min()}")
print(f"Máximo:     {df['final_exam_score'].max()}")

Media:      83.54
Mediana:    83.80
Desv. est.: 10.34
Mínimo:     46.8
Máximo:     100.0


### 4.2 Distribución de variables numéricas predictoras

In [11]:
# Variables numéricas predictoras (sin student_id ni final_exam_score)
cols_numericas = ['study_time_hours', 'attendance_percent', 'sleep_hours', 'previous_grade']

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, col in zip(axes, cols_numericas):
    ax.hist(df[col].dropna(), bins=20, edgecolor='black', alpha=0.7, color='teal')
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

### 4.3 Distribución de variables categóricas

In [12]:
cols_categoricas = ['gender', 'parental_education', 'internet_access',
                    'extracurricular_activities', 'part_time_job']

fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for ax, col in zip(axes, cols_categoricas):
    conteos = df[col].value_counts()
    ax.bar(conteos.index.astype(str), conteos.values, color='coral', edgecolor='black', alpha=0.8)
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 4.4 Matriz de correlación

In [13]:
cols_corr = cols_numericas + ['final_exam_score']
corr_matrix = df[cols_corr].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', square=True)
plt.title('Matriz de correlación — variables numéricas')
plt.tight_layout()
plt.show()

print("\nCorrelación con final_exam_score:")
print(corr_matrix['final_exam_score'].drop('final_exam_score').sort_values(ascending=False).round(4))


Correlación con final_exam_score:
study_time_hours      0.5675
previous_grade        0.4063
attendance_percent    0.2619
sleep_hours           0.1478
Name: final_exam_score, dtype: float64


### 4.5 Relación entre predictores numéricos y la variable objetivo

In [14]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, col in zip(axes, cols_numericas):
    ax.scatter(df[col], df['final_exam_score'], alpha=0.3, s=15, color='steelblue')
    ax.set_xlabel(col)
    ax.set_ylabel('final_exam_score')
    ax.set_title(f'{col} vs final_exam_score')

plt.tight_layout()
plt.show()

### 4.6 Variables categóricas vs variable objetivo

In [15]:
fig, axes = plt.subplots(1, 5, figsize=(24, 5))

for ax, col in zip(axes, cols_categoricas):
    df.boxplot(column='final_exam_score', by=col, ax=ax)
    ax.set_title(f'final_exam_score por {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('final_exam_score')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('')
plt.tight_layout()
plt.show()

### 4.7 Distribución de calificaciones (final_grade)

In [16]:
conteo_grades = df['final_grade'].value_counts().sort_index()

plt.figure(figsize=(6, 4))
colores = {'A': '#2ecc71', 'B': '#3498db', 'C': '#f39c12', 'D': '#e74c3c', 'F': '#8e44ad'}
plt.bar(conteo_grades.index, conteo_grades.values,
        color=[colores.get(g, 'gray') for g in conteo_grades.index],
        edgecolor='black', alpha=0.85)
plt.xlabel('Calificación')
plt.ylabel('Cantidad de estudiantes')
plt.title('Distribución de calificaciones (final_grade)')
plt.tight_layout()
plt.show()

print(conteo_grades)

final_grade
A    284
B    354
C    261
D     89
F     12
Name: count, dtype: int64


### 4.8 Limitaciones del conjunto de datos

- **Tamaño moderado:** 1000 observaciones y 9 variables predictoras. Es suficiente para el alcance del proyecto, pero limita lo que puede aprender un modelo con muchos parámetros.
- **La variable objetivo está acotada en 100** y 78 estudiantes (7.8%) se sitúan exactamente en ese tope. El puntaje está censurado por arriba, de modo que el modelo no puede acertar por encima del límite y los residuos se concentran del lado negativo en la parte alta de la escala.
- **Relaciones moderadas:** la correlación más alta con el objetivo es 0.57 (horas de estudio semanales) y el resto son moderadas o débiles. Eso marca un techo previsible para cualquier modelo entrenado con estas variables.
- **Sin dimensión temporal:** cada fila es una fotografía de un estudiante, no una serie. No permite observar la evolución de un mismo estudiante ni predecir con información acumulada a lo largo del curso.

---

## 5. Preparación de los datos

### 5.1 Definir variables predictoras y variable objetivo

**Prevención de fuga de información:**
- Se excluye `final_grade` porque se deriva directamente de `final_exam_score` (es la nota en letras del mismo examen). Incluirla causaría **data leakage**.
- Se excluye `student_id` porque es solo un identificador sin valor predictivo.
- Se conserva `previous_grade` porque es una calificación **previa**, no derivada del target.
- Se conserva `parental_education` con sus cinco niveles, incluido `None`, según lo analizado en la sección 3.

In [17]:
# Definir predictoras y target
cols_a_excluir = ['student_id', 'final_exam_score', 'final_grade']
feature_cols = [c for c in df.columns if c not in cols_a_excluir]

X = df[feature_cols].copy()
y = df['final_exam_score'].copy()

print(f"Variables predictoras ({len(feature_cols)}):")
for col in feature_cols:
    print(f"  - {col:30s} ({X[col].dtype})")
print(f"\nVariable objetivo: final_exam_score")
print(f"\nForma de X: {X.shape}")
print(f"Forma de y: {y.shape}")

Variables predictoras (9):
  - gender                         (str)
  - study_time_hours               (float64)
  - attendance_percent             (float64)
  - sleep_hours                    (float64)
  - parental_education             (str)
  - internet_access                (str)
  - extracurricular_activities     (str)
  - part_time_job                  (str)
  - previous_grade                 (float64)

Variable objetivo: final_exam_score

Forma de X: (1000, 9)
Forma de y: (1000,)


### 5.2 División train/test

Se reserva el 20% de los registros para prueba mediante `train_test_split` con semilla fija (`random_state=42`), en una división aleatoria simple. Cada fila corresponde a un estudiante distinto, de modo que no existen grupos naturales que debieran permanecer juntos en la misma partición y la división aleatoria no introduce fuga de información.

La separación se realiza **antes** de cualquier transformación: el escalado y la codificación se ajustan únicamente con los datos de entrenamiento.

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Conjunto de entrenamiento: {X_train.shape[0]} muestras ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Conjunto de prueba:        {X_test.shape[0]} muestras ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"\nNulos en train (parental_education): {X_train['parental_education'].isnull().sum()}")
print(f"Nulos en test  (parental_education): {X_test['parental_education'].isnull().sum()}")

Conjunto de entrenamiento: 800 muestras (80%)
Conjunto de prueba:        200 muestras (20%)

Nulos en train (parental_education): 0
Nulos en test  (parental_education): 0


### 5.3 Pipeline de preprocesamiento

Usamos `ColumnTransformer` de scikit-learn para construir un pipeline que escala las variables numéricas con `StandardScaler` y codifica las categóricas con `OneHotEncoder`.

No incluye imputación porque el dataset no tiene valores ausentes, como quedó verificado en la sección 3. `handle_unknown='ignore'` permite que el modelo reciba categorías no vistas durante el entrenamiento sin fallar.

El pipeline garantiza que **no hay fuga de información**: su `.fit()` se ejecuta solo sobre los datos de entrenamiento.

In [19]:
# Identificar columnas numéricas y categóricas
num_features = ['study_time_hours', 'attendance_percent', 'sleep_hours', 'previous_grade']
cat_features = ['gender', 'parental_education', 'internet_access',
                'extracurricular_activities', 'part_time_job']

print(f"Features numéricas ({len(num_features)}): {num_features}")
print(f"Features categóricas ({len(cat_features)}): {cat_features}")

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
    ]
)

print("\nPipeline de preprocesamiento creado correctamente.")

Features numéricas (4): ['study_time_hours', 'attendance_percent', 'sleep_hours', 'previous_grade']
Features categóricas (5): ['gender', 'parental_education', 'internet_access', 'extracurricular_activities', 'part_time_job']

Pipeline de preprocesamiento creado correctamente.


## 6. Modelo base

Antes de entrenar cualquier modelo se construye un **modelo base** (`DummyRegressor`) que predice siempre la media del puntaje en el conjunto de entrenamiento. No pretende ser bueno: es el punto de comparación que permite saber si el modelo predictivo aporta algo sobre no modelar nada.

In [20]:
modelo_base = DummyRegressor(strategy='mean')
modelo_base.fit(X_train, y_train)
y_pred_base = modelo_base.predict(X_test)

mae_base  = mean_absolute_error(y_test, y_pred_base)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_base))
r2_base   = r2_score(y_test, y_pred_base)

print(f"Predicción constante del modelo base: {modelo_base.constant_[0][0]:.2f} puntos")
print(f"\n  MAE  {mae_base:.4f}")
print(f"  RMSE {rmse_base:.4f}")
print(f"  R²   {r2_base:.4f}")

Predicción constante del modelo base: 83.87 puntos

  MAE  8.5847
  RMSE 10.5013
  R²   -0.0245


## 7. Entrenamiento y comparación de modelos

Se evalúan seis modelos de regresión con validación cruzada de cinco particiones sobre el conjunto de entrenamiento. Se emplea validación cruzada en lugar de una única partición de validación porque con 800 registros una sola división daría una estimación inestable; promediar cinco particiones reduce esa varianza y, sobre todo, permite leer la desviación entre particiones, que es lo que revela si la diferencia entre dos modelos es real o es ruido.

El conjunto de prueba no interviene en esta comparación: queda reservado para la evaluación final.

In [21]:
modelos = {
    'Linear Regression': LinearRegression(),
    'Ridge':             Ridge(alpha=1.0),
    'Lasso':             Lasso(alpha=0.1),
    'Decision Tree':     DecisionTreeRegressor(max_depth=10, random_state=42),
    'Random Forest':     RandomForestRegressor(n_estimators=100, max_depth=15,
                                               random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5,
                                                    random_state=42),
}

resultados = {}

print(f"{'Modelo':25s}  {'R² (CV)':>15s}  {'MAE (CV)':>15s}")
print("-" * 60)

for nombre, modelo in modelos.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', modelo)
    ])

    scores_r2 = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='r2')
    scores_mae = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')

    resultados[nombre] = {
        'R2_mean': scores_r2.mean(),
        'R2_std': scores_r2.std(),
        'MAE_mean': -scores_mae.mean(),
        'MAE_std': scores_mae.std(),
    }

    print(f"{nombre:25s}  {scores_r2.mean():.4f} (±{scores_r2.std():.4f})  {-scores_mae.mean():.4f} (±{scores_mae.std():.4f})")

# Tabla resumen ordenada
df_resultados = pd.DataFrame(resultados).T.sort_values('R2_mean', ascending=False)
print("\n=== Ranking por R² ===")
print(df_resultados[['R2_mean', 'MAE_mean']].round(4))

Modelo                             R² (CV)         MAE (CV)
------------------------------------------------------------
Linear Regression          0.6617 (±0.0350)  4.7812 (±0.2780)


Ridge                      0.6618 (±0.0349)  4.7809 (±0.2782)


Lasso                      0.6631 (±0.0342)  4.7823 (±0.2909)


Decision Tree              0.2087 (±0.0998)  7.0450 (±0.3852)


Random Forest              0.5697 (±0.0423)  5.3383 (±0.3177)


Gradient Boosting          0.5712 (±0.0622)  5.3518 (±0.4712)

=== Ranking por R² ===
                   R2_mean  MAE_mean
Lasso               0.6631    4.7823
Ridge               0.6618    4.7809
Linear Regression   0.6617    4.7812
Gradient Boosting   0.5712    5.3518
Random Forest       0.5697    5.3383
Decision Tree       0.2087    7.0450


In [22]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

nombres = df_resultados.index
r2_vals = df_resultados['R2_mean']
mae_vals = df_resultados['MAE_mean']

axes[0].barh(nombres, r2_vals, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].set_xlabel('R² (validación cruzada)')
axes[0].set_title('Comparación de modelos — R²')

axes[1].barh(nombres, mae_vals, color='coral', edgecolor='black', alpha=0.8)
axes[1].set_xlabel('MAE (validación cruzada)')
axes[1].set_title('Comparación de modelos — MAE')

plt.tight_layout()
plt.show()

## 8. Modelo seleccionado — Entrenamiento final

Los tres modelos lineales quedan empatados: la diferencia de R² entre el primero y el tercero es de milésimas, muy por debajo de la desviación de la propia validación cruzada. Elegir automáticamente el de mayor R² haría que el modelo entregado cambiara con la semilla del split, y la Fase 2 necesita un modelo determinista.

Por eso se fija **Lasso**: rinde igual que Ridge y que la regresión lineal, y además anula los coeficientes irrelevantes, con lo que es el más simple de los tres equivalentes.

In [23]:
# El algoritmo se fija explícitamente: no se elige en tiempo de ejecución
NOMBRE_MODELO = 'Lasso'
modelo_final = Lasso(alpha=0.1)

print(f"Modelo seleccionado: {NOMBRE_MODELO}")
print(f"R² promedio en CV:   {df_resultados.loc[NOMBRE_MODELO, 'R2_mean']:.4f}")
print(f"MAE promedio en CV:  {df_resultados.loc[NOMBRE_MODELO, 'MAE_mean']:.4f}")

pipeline_final = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', modelo_final)
])

pipeline_final.fit(X_train, y_train)
print(f"\nModelo entrenado con {X_train.shape[0]} muestras.")

Modelo seleccionado: Lasso
R² promedio en CV:   0.6631
MAE promedio en CV:  4.7823

Modelo entrenado con 800 muestras.


## 9. Evaluación en el conjunto de prueba

**Métrica principal: MAE.** El error absoluto medio se expresa en la misma unidad que la variable objetivo —puntos del examen, sobre 100—, de modo que su valor se interpreta directamente: cuántos puntos se equivoca el modelo en promedio. Se reportan además RMSE, que penaliza más los errores grandes, y R², que indica qué proporción de la variabilidad del puntaje explica el modelo.

In [24]:
y_pred = pipeline_final.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print("=" * 58)
print("   EVALUACIÓN EN EL CONJUNTO DE PRUEBA")
print("=" * 58)
print(f"{'':22s}{'Modelo base':>15s}{NOMBRE_MODELO:>15s}")
print("-" * 58)
print(f"{'MAE':22s}{mae_base:>15.4f}{mae:>15.4f}")
print(f"{'RMSE':22s}{rmse_base:>15.4f}{rmse:>15.4f}")
print(f"{'R²':22s}{r2_base:>15.4f}{r2:>15.4f}")
print("=" * 58)
print(f"El modelo reduce el MAE un {(1 - mae/mae_base)*100:.1f}% respecto al modelo base.")

   EVALUACIÓN EN EL CONJUNTO DE PRUEBA
                          Modelo base          Lasso
----------------------------------------------------------
MAE                            8.5847         4.9360
RMSE                          10.5013         6.3659
R²                            -0.0245         0.6235
El modelo reduce el MAE un 42.5% respecto al modelo base.


In [25]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Real vs Predicho
axes[0].scatter(y_test, y_pred, alpha=0.5, s=20, color='steelblue')
lim_min = min(y_test.min(), y_pred.min()) - 5
lim_max = max(y_test.max(), y_pred.max()) + 5
axes[0].plot([lim_min, lim_max], [lim_min, lim_max], 'r--', lw=2, label='Predicción perfecta')
axes[0].set_xlabel('Valor real')
axes[0].set_ylabel('Valor predicho')
axes[0].set_title('Real vs Predicho')
axes[0].legend()

# 2. Distribución de residuos
residuos = y_test.values - y_pred
axes[1].hist(residuos, bins=25, edgecolor='black', alpha=0.7, color='teal')
axes[1].set_xlabel('Residuo (real - predicho)')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de residuos')
axes[1].axvline(0, color='red', linestyle='--')

# 3. Residuos vs Predicho
axes[2].scatter(y_pred, residuos, alpha=0.5, s=20, color='coral')
axes[2].axhline(0, color='red', linestyle='--')
axes[2].set_xlabel('Valor predicho')
axes[2].set_ylabel('Residuo')
axes[2].set_title('Residuos vs Predicho')

plt.tight_layout()
plt.show()

## 10. Importancia de características

Lasso no entrega importancias como un modelo de árbol, sino coeficientes. Las variables numéricas están estandarizadas, así que su coeficiente indica cuántos puntos cambia la predicción por cada desviación estándar; en las categóricas codificadas indica el efecto de pertenecer a esa categoría.

In [26]:
modelo_entrenado = pipeline_final.named_steps['model']

# El ColumnTransformer ya expone los nombres de las columnas que produce
feature_names_final = [n.split('__', 1)[1]
                       for n in pipeline_final.named_steps['preprocessor'].get_feature_names_out()]

coefs = modelo_entrenado.coef_
df_coefs = pd.DataFrame({
    'Feature': feature_names_final,
    'Coeficiente': coefs
}).sort_values('Coeficiente', key=abs, ascending=True)

plt.figure(figsize=(10, max(6, len(df_coefs) * 0.35)))
colores = ['coral' if c < 0 else 'steelblue' for c in df_coefs['Coeficiente']]
plt.barh(df_coefs['Feature'], df_coefs['Coeficiente'], color=colores, edgecolor='black')
plt.xlabel('Coeficiente (sobre variables estandarizadas)')
plt.title(f'Coeficientes del modelo — {NOMBRE_MODELO}')
plt.tight_layout()
plt.show()

print("Variables con mayor peso en la predicción:")
for _, row in df_coefs.iloc[::-1].head(5).iterrows():
    print(f"  {row['Feature']:35s}  {row['Coeficiente']:+.4f}")

anuladas = df_coefs[df_coefs['Coeficiente'] == 0]['Feature'].tolist()
print(f"\nVariables anuladas por Lasso ({len(anuladas)}): {anuladas}")

Variables con mayor peso en la predicción:
  study_time_hours                     +5.9388
  internet_access_No                   -5.1711
  previous_grade                       +4.2524
  part_time_job_No                     +3.0973
  attendance_percent                   +3.0382

Variables anuladas por Lasso (8): ['parental_education_High School', 'parental_education_Bachelors', 'gender_Male', 'gender_Female', 'internet_access_Yes', 'parental_education_PhD', 'parental_education_None', 'parental_education_Masters']


## 11. Guardar el modelo

Guardamos el pipeline completo (preprocesamiento + modelo) con `joblib` para reutilizarlo en las fases 2, 3 y 4 del proyecto.

In [27]:
os.makedirs('modelo', exist_ok=True)

modelo_path = 'modelo/pipeline_modelo.joblib'
joblib.dump(pipeline_final, modelo_path)

file_size = os.path.getsize(modelo_path) / 1024
print(f"Modelo guardado en: {modelo_path}")
print(f"Tamaño del archivo: {file_size:.1f} KB")

Modelo guardado en: modelo/pipeline_modelo.joblib
Tamaño del archivo: 4.3 KB


### Verificación: cargar y usar el modelo guardado

In [28]:
# Cargar el modelo
pipeline_cargado = joblib.load(modelo_path)

# Verificar que produce las mismas predicciones
y_pred_check = pipeline_cargado.predict(X_test)
diferencia = np.abs(y_pred - y_pred_check).max()
print(f"Diferencia máxima entre predicciones: {diferencia}")
print(f"Verificación: {'✅ OK — Modelo cargado correctamente' if diferencia < 1e-10 else '❌ ERROR'}")

# Ejemplo con datos nuevos
print("\n--- Ejemplo de predicción con datos nuevos ---")
ejemplo = pd.DataFrame({
    'study_time_hours': [5.0],
    'attendance_percent': [90.0],
    'sleep_hours': [7.0],
    'previous_grade': [80.0],
    'gender': ['Female'],
    'parental_education': ['Bachelors'],
    'internet_access': ['Yes'],
    'extracurricular_activities': ['Yes'],
    'part_time_job': ['No']
})
pred = pipeline_cargado.predict(ejemplo)
print(f"Estudiante de ejemplo → Puntaje predicho: {pred[0]:.1f}")

Diferencia máxima entre predicciones: 0.0
Verificación: ✅ OK — Modelo cargado correctamente

--- Ejemplo de predicción con datos nuevos ---
Estudiante de ejemplo → Puntaje predicho: 96.7


## 12. Resumen y conclusiones

### Resultados

| | Modelo base | Lasso |
|---|---|---|
| MAE | 8.5847 | **4.9360** |
| RMSE | 10.5013 | **6.3659** |
| R² | −0.0245 | **0.6235** |

**¿El modelo mejora al modelo base?** Sí. Reduce el error absoluto medio un 42.5%: predecir siempre la media se equivoca en 8.6 puntos y el modelo en 4.9. El R² pasa de prácticamente cero a 0.62, es decir que el modelo explica el 62% de la variabilidad del puntaje.

**¿La métrica obtenida es razonable?** Sí para el propósito del proyecto. Un error medio de 4.9 puntos sobre una escala de 100, con puntajes que van de 46.8 a 100, permite distinguir a un estudiante en riesgo de uno que no lo está, que es el uso previsto. No alcanza para predecir la nota exacta de un estudiante concreto.

### Qué pesa en la predicción

Las horas de estudio semanales son la variable dominante (+5.94 puntos por desviación estándar), seguidas de la falta de acceso a internet (−5.17), la calificación previa (+4.25), no tener trabajo de medio tiempo (+3.10) y la asistencia (+3.04).

Lasso anuló ocho coeficientes: los cinco niveles de `parental_education` y los dos de `gender`. Con estos datos, la educación de los padres y el género no aportan nada a la predicción una vez que se conocen los hábitos de estudio.

### Dificultades encontradas

1. **Los valores faltantes no eran faltantes.** Los 102 nulos de `parental_education` resultaron ser la cadena `None`, que pandas convierte automáticamente a nulo, y que en esta variable significa *sin educación formal*. Detectarlo cambió la estrategia de preparación: de imputar con la moda a conservar la categoría.
2. **Los tres modelos lineales quedaron empatados.** Sus R² difieren en milésimas, muy por debajo de la desviación de la validación cruzada, de modo que elegir automáticamente el mejor entregaba un modelo distinto según la semilla. Se optó por fijar el algoritmo.
3. **El puntaje está topado en 100** y 78 estudiantes (7.8%) están exactamente en el tope. El modelo no puede superar ese límite, lo que concentra residuos negativos en la parte alta de la escala.

### Qué mejorar en futuras versiones

- Ajustar `alpha` de Lasso por validación cruzada en lugar de dejarlo fijo en 0.1.
- Acotar las predicciones al rango válido de la escala, que hoy el modelo puede rebasar.
- Evaluar si conviene retirar del modelo las variables cuyo coeficiente quedó en cero.